# Qualitative Analysis (Scan)

This notebook demonstrates a practical workflow for analyzing **scan-mode** MS data in an interactive Python environment.
**QualAnalysis** is a Hive data object that represents mass spectra acquired in scan mode.

You will use:

- `hive_data_access` to open a Hive file and access data objects
- `hivetoolkit` to visualize chromatograms and spectra


## Preparation

### Importing required libraries

This notebook assumes that `hive-data-access` and `hivetoolkit` are available in the current Python environment.  
When you install `hivetoolkit` using `pip install ./hivetoolkit`, the `hive-data-access` package will also be installed from PyPI.

Note: Reading `.hmD` data may require a Hive Runtime subscription or license, whereas `.mzB` files are typically readable without license verification.


In [ ]:
import hive_data_access
import hivetoolkit

## File Access

Open a sample file as a `HiveFile` object. This is the entry point for accessing measurements and analyses via `hive_data_access`.


In [ ]:
from pathlib import Path

# set the file path to a Hive-format file
qual_file_path = Path('../sample/sample_qual.mzB')

# open hive file
hive = hive_data_access.HiveFile(qual_file_path)

## Measurement

A `Measurement` represents one sample within a file. Most files contain a single measurement, but some vendor formats can contain multiple.

- Check how many measurements exist: `hive.get_measurement_count()`
- Retrieve one measurement: `hive.get_measurement(i)` (i: zero-based index)

A `Measurement` also provides metadata such as sample name and acquisition time.


In [ ]:
# check the number of measurements
print('Measurement Count: ' + str(hive.get_measurement_count()))

# select first measurement
measurement = hive.get_measurement(0)

# show part of the measurement information as an example
print('Sample Name: ' + str(measurement.sample_name))
print('Measurement Date Time: ' + str(measurement.measurement_date_time))
print('Injection Volume: ' + str(measurement.injection_volume))
print('Plate Code: ' + str(measurement.plate_code))
print('Well Code: ' + str(measurement.well_code))

## QualAnalysis

Scan-mode spectra are provided as a `QualAnalysis` object under a `Measurement`.

- Confirm how many analyses exist: `measurement.get_qual_analysis_count()`
- Retrieve the object: `measurement.get_qual_analysis(i)` (i: zero-based index)

In scan mode, spectra acquired under different acquisition conditions (e.g., polarity or scan events) are stored together in the order of acquisition.
When selecting spectra or building chromatograms, make sure you use the intended acquisition condition.


In [ ]:
# check the number of QualAnalysis
print('QualAnalysis Count: ' + str(measurement.get_qual_analysis_count()))

# select first QualAnalysis
qual = measurement.get_qual_analysis(0)

# check the number of acquisition conditions within the QualAnalysis.
# also check the contents of the first 5 conditions, to see which condition should be selected.
# selection of the appropriate acquisition condition is crucial for the following analysis, to observe consistent scan data.
print('Number of unique acquisition conditions: ' + str(len(qual.unique_acquisition_conditions)))
print("\n".join(map(str, [qual.unique_acquisition_conditions[n] for n in range(min(5, len(qual.unique_acquisition_conditions)))])))

# select first analysis condition
condition = qual.unique_acquisition_conditions[0]

### Chromatogram & Spectrum

Typical chromatograms derived from scan MS data include TIC, BPC, and XIC.

From a `QualAnalysis`, you can obtain chromatogram point lists using:
- `get_tic(acquisition_condition)`
- `get_bpc(acquisition_condition)`
- `get_xic(filter_option)`
- `get_multiple_xic(filter_options)`

These methods require an `AcquisitionCondition` (directly or inside the filter option) to ensure consistency.

The returned point list(s) can be passed to `hivetoolkit.draw_chromatogram(...)`.

To retrieve a spectrum, use:
- `get_spectrum(i)` (i: zero-based index)
- `find_nearest_spectrum(retention_time)`

The retrieved spectra can be visualized using `hivetoolkit.draw_spectrum(...)`.

For an overview of spectral changes across a chromatographic run, multiple spectra can be visualized together using `hivetoolkit.draw_spectra_map(...)`.

In [ ]:
# draw spectra map
# The spectra map helps you get a rough overview of the entire scan data and identify significant peaks for further detailed examination.
hivetoolkit.draw_spectra_map(qual, condition)

In [ ]:
# get a chromatogram, which is a list of RT and intensity pairs
points = qual.get_tic(condition)
# points = qual.get_bpc(condition) # for basepeak chromatogram

# draw a chromatogram (title and labels arguments are optional)
# The "labels" argument specifies the location and format of the label strings.
# This example labels the top 5 peaks with their RT values (x), formatted to two decimal places.
hivetoolkit.draw_chromatogram(points, title = 'Total Ion Chromatogram', labels=dict(top=dict(n=5, format='{x:.2f}')))

In [ ]:
# defines the extract conditions for acquiring XIC
# This example examines a significant peak at m/z=780, RT around 9 min.
extract_mz_range = hive_data_access.ValueRange(779, 781) # m/z range around 780 +- 1
extract_rt_range = hive_data_access.ValueRange(7, 11) # range of RT in minutes

# set the above conditions as a filter (required only for XIC)  N.B. the last extract_rt_range argument is optional.
chromatogram_filter_option = hive_data_access.ChromatogramFilterOption(condition, extract_mz_range, extract_rt_range)

# get a chromatogram, which is a list of RT and intensity pairs
points = qual.get_xic(chromatogram_filter_option)

# draw a chromatogram
# This example labels the top 2 peaks with the default format of (RT, Intensity)
hivetoolkit.draw_chromatogram(points, title = 'eXtract Ion Chromatogram [m/z 779-781]', labels=dict(top=2))

In [ ]:
# select the nearest spectrum at 8.3 min.
spectrum = qual.find_nearest_spectrum(8.3)
# If the found spectrum's acquisition_condition is not the target condition, then take the next scan until the condition is found.
while spectrum.acquisition_condition != condition:
    spectrum = qual.get_spectrum(spectrum.index + 1)

# show a part of spectrum information (Others: collision energy, precursor value/range, spectrum type ...)
print('RT: ' + str(spectrum.retention_time))
condition = spectrum.acquisition_condition
print('MS-Level: ' + str(condition.ms_level))
print('Polarity: ' + str(condition.polarity))

In [ ]:
# get spectrum points
points = spectrum.get_spectrum_points() # whole spectrum points
points_zoom = [p for p in points if p.x > 750 and p.x < 830] # spectrum points of which m/z range are limited to between 750 and 830.

# draw spectrum
# 'color' and 'style' arguments are also optional, as well as 'title' and 'labels'.
# You can add the 'style' argument for each draw_... method, to override the default style.
# The 'offset' parameter in 'labels' means to draw label strings with 'offset' pixel points upper to the original location.
hivetoolkit.draw_spectrum(points, title='Spectrum [RT:8.3 min.] of whole range', color='darkred', labels=dict(top=dict(n=5, format="{x:.3f}"), offset=10))
hivetoolkit.draw_spectrum(points_zoom, title='Spectrum [RT:8.3 min.] of Zoomed m/z [750, 810]', color='darkred', labels=dict(top=dict(n=9, format="{x:.3f}"), offset=10))

## Termination

When you are done, close the file handle:

- `hive.close()`


In [ ]:
# close opened file
hive.close()